In [ ]:
# Configure the repository root once so the same notebook works locally and on Kaggle.
import os
import sys
from pathlib import Path

_configured_root = os.environ.get("DATA_DIR")
if _configured_root:
    DATA_DIR = Path(_configured_root).expanduser().resolve()
else:
    DATA_DIR = next(
        (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
         if (p / "README.md").is_file() and (p / "data").is_dir()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError("Set DATA_DIR to the repository root before running this notebook.")
os.environ["DATA_DIR"] = str(DATA_DIR)
os.chdir(DATA_DIR)
if str(DATA_DIR) not in sys.path:
    sys.path.insert(0, str(DATA_DIR))


In [ ]:
import copy
import hashlib
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import sys

# Set seed cố định
def set_seed(s=42):
  random.seed(s)
  np.random.seed(s)
  torch.manual_seed(s)
  torch.cuda.manual_seed_all(s)
set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sử dụng thiết bị: {DEVICE}")
print("[Info] PyTorch:", torch.__version__)
print("[Info] CUDA khả dụng:", torch.cuda.is_available())

sys.path.append(str(DATA_DIR / "src/eval")) 
from evaluator import evaluate

Sử dụng thiết bị: cuda
[Info] PyTorch: 2.11.0+cu128
[Info] CUDA khả dụng: True


In [ ]:
# Nạp các dữ liệu cần thiết 
raw_course_text_embeddings = np.load(str(DATA_DIR / "data/course_llm/1_embeddings/raw_course_text_embeddings.npy"))
print(f'Kích thước của embedding văn bản gốc: {raw_course_text_embeddings.shape}')

with open(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/item_index.json"), 'r') as f:
    course_ids = json.load(f)
assert len(course_ids) == 3769 and course_ids == sorted(course_ids), "Lỗi item_index.json"
assert course_ids == json.load(open(str(DATA_DIR / "data/course_llm/1_embeddings/raw_embeddings_course_index.json")))["course_id"]

with open(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/user_index_cold.json"), 'r') as f:
    user_cold_ids = json.load(f)
    
print(f'Tổng số khóa học: {len(course_ids)}, Tổng số học viên (khóa lạnh): {len(user_cold_ids)}')

g_i_cold = np.load(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/g_i.npy"))
g_u_cold = np.load(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/g_u.npy"))
print(f"Bảng kích thước: g_u_cold = {g_u_cold.shape}, g_i_cold = {g_i_cold.shape}")

assert g_u_cold.shape == (len(user_cold_ids), 64) and g_i_cold.shape == (len(course_ids), 64), "Lỗi kích thước embedding nền chung"

# In MD5 của backbone và embedding văn bản gốc
def file_md5(path):
    return hashlib.md5(open(path, 'rb').read()).hexdigest()

print(f"MD5 g_u.npy: {file_md5(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/g_u.npy"))}")
print(f"MD5 g_i.npy: {file_md5(str(DATA_DIR / "data/embeddings/01_lightgcn_backbone_cold/g_i.npy"))}")
print(f"MD5 raw_course_text_embeddings.npy: {file_md5(str(DATA_DIR / "data/course_llm/1_embeddings/raw_course_text_embeddings.npy"))}")

Kích thước của embedding văn bản gốc: (3769, 384)
Tổng số khóa học: 3769, Tổng số học viên (khóa lạnh): 98220
Bảng kích thước: g_u_cold = (98220, 64), g_i_cold = (3769, 64)
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7
MD5 raw_course_text_embeddings.npy: f80445649ef8a8217d048bd3e7ac0ad5


In [ ]:
# Chuẩn bị dữ liệu đánh giá tập test_cold
te_cold = pd.read_parquet(str(DATA_DIR / "data/splits/cold_scenario/test_cold.parquet"))
tr_cold = pd.read_parquet(str(DATA_DIR / "data/splits/cold_scenario/train_cold.parquet"))
va_cold = pd.read_parquet(str(DATA_DIR / "data/splits/cold_scenario/val_cold.parquet"))

user2id = {str(u): i for i, u in enumerate(user_cold_ids)}
item2id = {str(c): i for i, c in enumerate(course_ids)}

def build_hist(df):
    hist_dict = {}
    for u, group in df.groupby('user_id'):
        u_str = str(u)
        if u_str in user2id:
            u_idx = user2id[u_str]
            item_idxs = [item2id[str(i)] for i in group['course_id'] if str(i) in item2id]
            if len(item_idxs) > 0:
                hist_dict[u_idx] = item_idxs
    return hist_dict

test_target_dict = build_hist(te_cold)
print(f'test_target_dict với {len(test_target_dict)} học viên.')

# Đặt tên riêng hist_tr (cho cold-val) và hist_trva (cho cold-test)
hist_tr = build_hist(tr_cold)
print(f'hist_tr (chỉ train) với {len(hist_tr)} học viên.')

hist_trva = build_hist(pd.concat([tr_cold, va_cold]))
print(f'hist_trva (train + val) với {len(hist_trva)} học viên.')

val_target_dict = build_hist(va_cold)
print(f"val_target_dict với {len(val_target_dict)} học viên.")

test_target_dict với 3340 học viên.
hist_tr (chỉ train) với 98220 học viên.
hist_trva (train + val) với 98220 học viên.
val_target_dict với 2092 học viên.


In [ ]:
with open(str(DATA_DIR / "data/splits/cold_scenario/cold_items_val.json"), 'r') as f:
    cold_val_ids = json.load(f)
with open(str(DATA_DIR / "data/splits/cold_scenario/cold_items_test.json"), 'r') as f:
    cold_test_ids = json.load(f)

cold_val_ids = cold_val_ids.get("cold_items", cold_val_ids) if isinstance(cold_val_ids, dict) else cold_val_ids
cold_test_ids = cold_test_ids.get("cold_items", cold_test_ids) if isinstance(cold_test_ids, dict) else cold_test_ids

cold_item_set = set(str(c) for c in cold_val_ids + cold_test_ids)
cold_idx = np.array(sorted(item2id[c] for c in cold_item_set if c in item2id))

# Kiểm tra assertion khóa lạnh bằng 0 - không có tín hiệu ID 
assert np.allclose(g_i_cold[cold_idx], 0.0), "Lỗi: Khóa lạnh phải có embedding g_i = 0 đối với mô hình thuần ID!"

# Kiểm tra chặt chẽ tính tách biệt của kịch bản cold-item
tr_items = set(tr_cold.course_id.astype(str))
cv, ct = set(map(str, cold_val_ids)), set(map(str, cold_test_ids))
assert not (cv & tr_items) and not (ct & tr_items) and not (cv & ct), "Lỗi: Rò rỉ khóa học lạnh!"
assert set(va_cold.course_id.astype(str)) <= cv
assert set(te_cold.course_id.astype(str)) <= ct
assert set(va_cold.user_id) <= set(tr_cold.user_id)
assert set(te_cold.user_id) <= set(tr_cold.user_id)
print("Đã vượt qua toàn bộ kiểm tra assert không rò rỉ dữ liệu!")

Đã vượt qua toàn bộ kiểm tra assert không rò rỉ dữ liệu!


In [ ]:
# Khởi tạo dữ liệu tensor cho DropoutNet
import torch.nn.functional as F

# Chuyển đổi các ma trận numpy sang torch.tensor và đưa lên thiết bị (GPU/CPU)
U_tensor = torch.tensor(g_u_cold, dtype=torch.float32, device=DEVICE)  # (98220, 64)
V_tensor = torch.tensor(g_i_cold, dtype=torch.float32, device=DEVICE)  # (3769, 64)

# Chuẩn hóa L2 và chuẩn bị 3 đầu vào nội dung (raw, llm, fused)
nz = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)
S_raw = nz(raw_course_text_embeddings.astype(np.float32))

# Đường dẫn tới file embedding hồ sơ LLM 
LLM_NPY = str(DATA_DIR / "data/course_llm/1_embeddings/llm_profile_embeddings.npy")
if os.path.exists(LLM_NPY):
    S_llm = nz(np.load(LLM_NPY).astype(np.float32))
    print(f"MD5 llm_embeddings.npy: {file_md5(LLM_NPY)}")
    CONTENTS = {
        "raw": S_raw,
        "llm": S_llm,
        "fused": np.concatenate([S_raw, S_llm], axis=1)
    }
else:
    print("[Cảnh báo] Chưa tìm thấy file LLM_NPY, hiện tại chỉ chạy với đầu vào 'raw'.")
    CONTENTS = {"raw": S_raw}

# Chỉ lấy 2.401 khóa học ấm có tương tác trong tr_cold để tính MSE
warm_np = np.unique(tr_cold.course_id.astype(str).map(item2id).to_numpy())   # 2.401 khóa
warm_cols = torch.tensor(warm_np, dtype=torch.long, device=DEVICE)
assert len(warm_cols) == 2401 and not (set(warm_cols.tolist()) & set(cold_idx.tolist())), "Lỗi tập warm_cols!"

# Điểm đích R_teacher chỉ tính 1 lần trên đúng 2.401 cột khóa học ấm có tương tác
R_teacher = U_tensor @ V_tensor[warm_cols].t()  # [98220, 2401]

print(f"Kích thước tensor U: {U_tensor.shape}")
print(f"Kích thước tensor V: {V_tensor.shape}")
print(f"Kích thước R_teacher (chỉ trên khóa ấm có tương tác): {R_teacher.shape}")
for k, v in CONTENTS.items():
    print(f"Đầu vào '{k}': kích thước {v.shape}")

MD5 llm_embeddings.npy: fc1819307be227a1544e1be0a4017393
Kích thước tensor U: torch.Size([98220, 64])
Kích thước tensor V: torch.Size([3769, 64])
Kích thước R_teacher (chỉ trên khóa ấm có tương tác): torch.Size([98220, 2401])
Đầu vào 'raw': kích thước (3769, 384)
Đầu vào 'llm': kích thước (3769, 384)
Đầu vào 'fused': kích thước (3769, 768)


In [7]:
# Định nghĩa lớp DropoutNetItemCold
class DropoutNetItemCold(nn.Module):
    def __init__(self, d=64, d_content=384, h=200):
        super().__init__()
        self.fU = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, d))
        self.fV = nn.Sequential(nn.Linear(d + d_content, h), nn.Tanh(), nn.Linear(h, d))
        
    def forward(self, u_pref, v_pref, v_content):
        # u_pref: embedding học viên từ LightGCN nền
        # v_pref: embedding khóa học từ LightGCN nền (sẽ bị dropout về 0 khi mô phỏng khóa lạnh)
        # v_content: embedding văn bản của khóa học
        return self.fU(u_pref), self.fV(torch.cat([v_pref, v_content], -1))

# In số lượng tham số cho đầu vào 384 chiều và 768 chiều
for dim_c in [384, 768]:
    tmp_m = DropoutNetItemCold(d=64, d_content=dim_c, h=200)
    n_params = sum(p.numel() for p in tmp_m.parameters() if p.requires_grad)
    print(f"Số tham số DropoutNetItemCold (d_content={dim_c}): {n_params:,}")

Số tham số DropoutNetItemCold (d_content=384): 128,528
Số tham số DropoutNetItemCold (d_content=768): 205,328


In [8]:
import itertools, random

# Bỏ 'lam' vì DropoutNet không dùng, giữ 60 tổ hợp (lr, p, wd)
space = dict(lr=[1e-4, 5e-4, 1e-3, 5e-3, 1e-2], 
             p=[0.1, 0.3, 0.5, 0.8],
             wd=[1e-5, 1e-4, 1e-3])

# Tạo lưới không gian tìm kiếm chuẩn
grid = [dict(zip(space, v)) for v in itertools.product(*space.values())]

# Lấy ngẫu nhiên đúng 20 cấu hình với seed 42 cho mọi mô hình 
trials = random.Random(42).sample(grid, 20) 
print(f"Đã khởi tạo thành công {len(trials)} cấu hình tinh chỉnh ngẫu nhiên từ {len(grid)} tổ hợp.")

Đã khởi tạo thành công 20 cấu hình tinh chỉnh ngẫu nhiên từ 60 tổ hợp.


In [9]:
cand_cold_indices = [item2id[str(c)] for c in cold_test_ids if str(c) in item2id]
cand_bool_mask_a = np.zeros(3769, dtype=bool)
if len(cand_cold_indices) > 0:
    cand_bool_mask_a[cand_cold_indices] = True

cand_cold_indices_val = [item2id[str(c)] for c in cold_val_ids if str(c) in item2id]
cand_bool_mask_a_val = np.zeros(3769, dtype=bool)
if len(cand_cold_indices_val) > 0:
    cand_bool_mask_a_val[cand_cold_indices_val] = True

cand_bool_mask_b = np.ones(3769, dtype=bool)

In [10]:
# Suy luận giữ nguyên V_tensor (khóa lạnh đã có sẵn g_i = 0)
@torch.no_grad()
def dn_vectors(model, content_t):
    model.eval()
    return model(U_tensor, V_tensor, content_t)

def dn_ndcg_val(model, content_t, seed):
    u_e, v_e = dn_vectors(model, content_t)
    r = evaluate(lambda ub: u_e[ub.to(DEVICE)] @ v_e.t(), val_target_dict, mask=hist_tr,
                 cand=cand_bool_mask_a_val, ks=(20,), tie_seed=seed, bs=1024)
    return float(r["NDCG@20"].mean())

def train_dn(content_t, cfg, seed, max_steps=1000, eval_every=10, patience=10):
    set_seed(seed)
    model = DropoutNetItemCold(d=64, d_content=content_t.shape[1], h=200).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=cfg["wd"])
    best, best_state, bad, best_step = -1.0, None, 0, 0
    
    for step in range(1, max_steps + 1):
        model.train()
        opt.zero_grad()
        # Chỉ dropout và tính MSE trên 2.401 cột khóa học ấm có tương tác (warm_cols)
        keep = (torch.rand(len(warm_cols), 1, device=DEVICE) > cfg["p"]).float()
        u_p, v_p = model(U_tensor, V_tensor[warm_cols] * keep, content_t[warm_cols])
        loss = F.mse_loss(u_p @ v_p.t(), R_teacher)
        loss.backward()
        opt.step()
        
        if step % eval_every == 0:
            v = dn_ndcg_val(model, content_t, seed)
            if v > best:
                best, best_state, bad, best_step = v, copy.deepcopy(model.state_dict()), 0, step
            else:
                bad += 1
                if bad >= patience:
                    break
                    
    model.load_state_dict(best_state)
    return model, best, best_step

# Chạy tinh chỉnh 20 cấu hình cho từng loại đầu vào trong CONTENTS
best_configs = {}
for name, S in CONTENTS.items():
    print(f"\n--- BẮT ĐẦU TINH CHỈNH 20 CẤU HÌNH CHO DROPOUTNET (Đầu vào: {name}) ---")
    content_t = torch.tensor(S, dtype=torch.float32, device=DEVICE)
    vals = []
    for idx, cfg in enumerate(trials):
        _, val_ndcg, best_st = train_dn(content_t, cfg, seed=42)
        vals.append(val_ndcg)
        print(f"[{name}] Trial [{idx+1}/20] | Config: {cfg} | Best step: {best_st} | NDCG@20 (Val): {val_ndcg:.4f}")
        
    best_idx = int(np.argmax(vals))
    best_configs[name] = trials[best_idx]
    print(f"==> [{name}] Cấu hình tốt nhất: {best_configs[name]} | NDCG@20 (Val) cao nhất: {vals[best_idx]:.4f}")


--- BẮT ĐẦU TINH CHỈNH 20 CẤU HÌNH CHO DROPOUTNET (Đầu vào: raw) ---
[raw] Trial [1/20] | Config: {'lr': 0.005, 'p': 0.3, 'wd': 0.0001} | Best step: 10 | NDCG@20 (Val): 0.0855
[raw] Trial [2/20] | Config: {'lr': 0.0001, 'p': 0.5, 'wd': 0.0001} | Best step: 60 | NDCG@20 (Val): 0.0659
[raw] Trial [3/20] | Config: {'lr': 0.0001, 'p': 0.1, 'wd': 0.0001} | Best step: 50 | NDCG@20 (Val): 0.0648
[raw] Trial [4/20] | Config: {'lr': 0.005, 'p': 0.8, 'wd': 0.001} | Best step: 230 | NDCG@20 (Val): 0.0797
[raw] Trial [5/20] | Config: {'lr': 0.0005, 'p': 0.3, 'wd': 0.001} | Best step: 10 | NDCG@20 (Val): 0.0597
[raw] Trial [6/20] | Config: {'lr': 0.0005, 'p': 0.3, 'wd': 1e-05} | Best step: 10 | NDCG@20 (Val): 0.0647
[raw] Trial [7/20] | Config: {'lr': 0.0005, 'p': 0.1, 'wd': 0.001} | Best step: 460 | NDCG@20 (Val): 0.0802
[raw] Trial [8/20] | Config: {'lr': 0.0001, 'p': 0.5, 'wd': 0.001} | Best step: 20 | NDCG@20 (Val): 0.0655
[raw] Trial [9/20] | Config: {'lr': 0.01, 'p': 0.5, 'wd': 0.001} | Best

In [11]:
# Đã khởi tạo riêng hist_tr và hist_trva ở Cell 4 để tránh ghi đè mặt nạ lịch sử.
print(f"Kiểm tra mặt nạ: hist_tr có {len(hist_tr)} học viên, hist_trva có {len(hist_trva)} học viên.")

Kiểm tra mặt nạ: hist_tr có 98220 học viên, hist_trva có 98220 học viên.


In [12]:
seeds_list = [42, 2023, 2024, 1007, 999]
os.makedirs("results_dropoutnet/item_reps", exist_ok=True)

all_results_a = {}
all_results_b = {}

for name, S in CONTENTS.items():
    print(f"\n--- BẮT ĐẦU ĐÁNH GIÁ 5 SEED CHO DROPOUTNET (Đầu vào: {name}) ---")
    content_t = torch.tensor(S, dtype=torch.float32, device=DEVICE)
    cfg = best_configs[name]
    
    results_a = []
    results_b = []
    user_ndcg20_seeds = []
    
    for s in seeds_list:
        model_seed, val_best, best_step = train_dn(content_t, cfg, seed=s)
        u_e, v_e = dn_vectors(model_seed, content_t)
        
        fn = lambda ub: u_e[ub.to(DEVICE)] @ v_e.t()
        kw = dict(mask=hist_trva, ks=(5, 10, 20), tie_seed=s, bs=1024)
        
        # 1. Đánh giá trên tập (a) - Khóa lạnh
        res_a = evaluate(fn, test_target_dict, cand=cand_bool_mask_a, **kw)
        results_a.append(res_a)
        user_ndcg20_seeds.append(res_a["NDCG@20"])
        
        # 2. Đánh giá trên tập (b) - Toàn bộ item
        res_b = evaluate(fn, test_target_dict, cand=cand_bool_mask_b, **kw)
        results_b.append(res_b)
        
        # Lưu u_e, v_e, best_step và NDCG@20 từng học viên
        np.save(f"results_dropoutnet/item_reps/u_e_{name}_seed{s}.npy", u_e.cpu().numpy())
        np.save(f"results_dropoutnet/item_reps/v_e_{name}_seed{s}.npy", v_e.cpu().numpy())
        np.save(f"results_dropoutnet/user_ndcg20_dropoutnet_{name}_seed{s}.npy", res_a["NDCG@20"])
        print(f"  Seed {s} | Best step (Val): {best_step} | Val NDCG@20: {val_best:.4f} | Test (a) Recall@20: {res_a['Recall@20'].mean():.4f}, NDCG@20: {res_a['NDCG@20'].mean():.4f} | Test (b) Recall@20: {res_b['Recall@20'].mean():.4f}")
        
    # Lưu điểm NDCG@20 trung bình 5 seed của từng học viên (3.340 học viên) để chạy Wilcoxon 
    mean_user_ndcg20 = np.mean(np.stack(user_ndcg20_seeds, axis=0), axis=0)
    np.save(f"user_ndcg20_dropoutnet_{name}.npy", mean_user_ndcg20)
    
    with open(f"results_dropoutnet/best_cfg_{name}.json", "w") as f:
        json.dump(cfg, f, indent=2)
        
    all_results_a[name] = results_a
    all_results_b[name] = results_b


--- BẮT ĐẦU ĐÁNH GIÁ 5 SEED CHO DROPOUTNET (Đầu vào: raw) ---
  Seed 42 | Best step (Val): 460 | Val NDCG@20: 0.0932 | Test (a) Recall@20: 0.2002, NDCG@20: 0.0854 | Test (b) Recall@20: 0.0003
  Seed 2023 | Best step (Val): 210 | Val NDCG@20: 0.0797 | Test (a) Recall@20: 0.1914, NDCG@20: 0.0807 | Test (b) Recall@20: 0.0000
  Seed 2024 | Best step (Val): 160 | Val NDCG@20: 0.0802 | Test (a) Recall@20: 0.1838, NDCG@20: 0.0847 | Test (b) Recall@20: 0.0000
  Seed 1007 | Best step (Val): 220 | Val NDCG@20: 0.0782 | Test (a) Recall@20: 0.1852, NDCG@20: 0.0856 | Test (b) Recall@20: 0.0000
  Seed 999 | Best step (Val): 200 | Val NDCG@20: 0.0802 | Test (a) Recall@20: 0.1830, NDCG@20: 0.0772 | Test (b) Recall@20: 0.0000

--- BẮT ĐẦU ĐÁNH GIÁ 5 SEED CHO DROPOUTNET (Đầu vào: llm) ---
  Seed 42 | Best step (Val): 150 | Val NDCG@20: 0.0995 | Test (a) Recall@20: 0.2109, NDCG@20: 0.0834 | Test (b) Recall@20: 0.0000
  Seed 2023 | Best step (Val): 70 | Val NDCG@20: 0.1061 | Test (a) Recall@20: 0.1753, N

In [13]:
for name in CONTENTS.keys():
    results_a = all_results_a[name]
    results_b = all_results_b[name]
    
    print(f"\n========== KẾT QUẢ TỔNG HỢP DROPOUTNET - ĐẦU VÀO: {name.upper()} (5 SEED) ==========")
    print("--- Tập (a): Khóa lạnh ---")
    for k in results_a[0].keys():
        seed_means = np.array([res[k].mean() for res in results_a])
        mean_value = seed_means.mean()
        std_value = seed_means.std(ddof=1)  
        print(f"{k}: {mean_value:.4f} ± {std_value:.4f}")

    print("\n--- Tập (b): Toàn bộ item ---")
    for k in results_b[0].keys():
        seed_means = np.array([res[k].mean() for res in results_b])
        mean_value = seed_means.mean()
        std_value = seed_means.std(ddof=1)  
        print(f"{k}: {mean_value:.4f} ± {std_value:.4f}")


========== KẾT QUẢ TỔNG HỢP DROPOUTNET - ĐẦU VÀO: RAW (5 SEED) ==========
--- Tập (a): Khóa lạnh ---
Recall@5: 0.0804 ± 0.0091
Recall@10: 0.1269 ± 0.0058
Recall@20: 0.1887 ± 0.0072
NDCG@5: 0.0506 ± 0.0055
NDCG@10: 0.0662 ± 0.0030
NDCG@20: 0.0827 ± 0.0037
HR@5: 0.0889 ± 0.0077
HR@10: 0.1395 ± 0.0072
HR@20: 0.2144 ± 0.0138

--- Tập (b): Toàn bộ item ---
Recall@5: 0.0000 ± 0.0000
Recall@10: 0.0000 ± 0.0000
Recall@20: 0.0001 ± 0.0001
NDCG@5: 0.0000 ± 0.0000
NDCG@10: 0.0000 ± 0.0000
NDCG@20: 0.0000 ± 0.0000
HR@5: 0.0000 ± 0.0000
HR@10: 0.0001 ± 0.0001
HR@20: 0.0001 ± 0.0003

========== KẾT QUẢ TỔNG HỢP DROPOUTNET - ĐẦU VÀO: LLM (5 SEED) ==========
--- Tập (a): Khóa lạnh ---
Recall@5: 0.0521 ± 0.0097
Recall@10: 0.1047 ± 0.0187
Recall@20: 0.1859 ± 0.0147
NDCG@5: 0.0334 ± 0.0062
NDCG@10: 0.0516 ± 0.0096
NDCG@20: 0.0737 ± 0.0077
HR@5: 0.0617 ± 0.0113
HR@10: 0.1289 ± 0.0221
HR@20: 0.2337 ± 0.0223

--- Tập (b): Toàn bộ item ---
Recall@5: 0.0000 ± 0.0000
Recall@10: 0.0000 ± 0.0000
Recall@20: 0.00